# Energy budgets

Compute a time series for the energy components and exchanges

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from scores.budgets import energy_components, energy_exchanges
import matplotlib.pyplot as plt

In [ ]:
START_TIME = pd.Timestamp("2020-01-01")
END_TIME = pd.Timestamp("2020-02-01")
ds = xr.open_zarr("gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr")
ds = ds.sel(time=slice(START_TIME, END_TIME))

Prepare the fields for the energy budget time series, using the zonal, meridional and vertical velocities, the temperature, water vapor, surface pressure and surface geopotential (the orography).

In [ ]:
fieldnames = ['u_component_of_wind','v_component_of_wind','vertical_velocity','temperature','specific_humidity','geopotential','surface_pressure','geopotential_at_surface']
ds = ds[fieldnames]
ds = ds.compute()

Specify a sub-domain of the global to compute the budgets over

In [ ]:
sub_domain_longitude = np.array([None,None])
sub_domain_latitude = np.array([-60.0,-20.0])

Compute a time series for the internal, latent, potential and horizontal and vertical kinetic energies:

Internal = $\int_{p_1}^{p_0}\int_{\Omega}(C_p^d(1-q) + C_p^vq)T\mathrm{d}\Omega\mathrm{d}p$ 

Latent = $\int_{p_1}^{p_0}\int_{\Omega}L_vq\mathrm{d}\Omega\mathrm{d}p$ 

Potential = $\int_{\Omega}z_s\Phi_s\mathrm{d}\Omega$ 

Kinetic (horizontal) = $\int_{p_1}^{p_0}\int_{\Omega}\frac{1}{2}(u^2 + v^2)\mathrm{d}\Omega\mathrm{d}p$ 

Kinetic (vertical) = $\int_{p_1}^{p_0}\int_{\Omega}\frac{1}{2}w^2\mathrm{d}\Omega\mathrm{d}p$ 

References: 
- Trenberth, K. E., Stepaniak, D. P., Caron, J. M. (2002) "Accuracy of Atmospheric Energy Budgets from Analyses" J. Clim. 15 3343--3360
- Sha, Y., Schreck, J. S., Chapman, W., Gagne, D. J. (2025) "Improving AI Weather Prediction Models using Global Mass and Energy Conservation Schemes" arXiv:2501.05648v2
- Taylor, M. A. (2011). Conservation of mass and energy for the moist atmospheric primitive equations on unstructured grids. In P. H.
Lauritzen, et al. (Eds.), Numerical techniques for global atmospheric models, Lecture Notes Comput. Sci. Eng. (Vol. 80, pp. 357–380).
Heidelberg, Germany: Springer.

In [ ]:
E = energy_components(ds, fieldnames, sub_domain_longitude, sub_domain_latitude)

In [ ]:
time=0.25*np.arange(len(E.time))
e_names = ['Internal','Latent','Potential','Kinetic (horiz.)','Kinetic (vert.)']
plt.plot(time,E["Internal"].data-E["Internal"].data[0])
plt.plot(time,E["Latent"].data-E["Latent"].data[0])
plt.plot(time,E["Potential"].data-E["Potential"].data[0])
plt.plot(time,E["HorizontalKinetic"].data-E["HorizontalKinetic"].data[0])
plt.plot(time,E["VerticalKinetic"].data-E["VerticalKinetic"].data[0])
plt.legend(e_names)
plt.title('change in energy from initial values')
plt.xlabel('time (days)')
plt.show()

Compute a time series for the kinetic to internal, internal to kinetic, kinetic to potential and potential to kinetic energy exchanges as:

Kinetic to Internal = $\int_{p_1}^{p_0}\int_{\Omega}\nabla(z-z_s)\cdot\boldsymbol{u}\mathrm{d}\Omega\mathrm{d}p$

Internal to Kinetic = $\int_{p_1}^{p_0}\int_{\Omega}(z-z_s)\nabla\cdot\boldsymbol{u}\mathrm{d}\Omega\mathrm{d}p$

Kinetic to Potential = $\int_{p_1}^{p_0}\int_{\Omega}\nabla z_s\cdot\boldsymbol{u}\mathrm{d}\Omega\mathrm{d}p$

Potential to Kinetic = $\int_{p_1}^{p_0}\int_{\Omega}z_s\nabla\cdot\boldsymbol{u}\mathrm{d}\Omega\mathrm{d}p$

In [ ]:
Ex = energy_exchanges(ds, fieldnames, sub_domain_longitude, sub_domain_latitude)

In [ ]:
time=0.25*np.arange(len(Ex.time))
ex_names = ['Kinetic to Internal','Internal to Kinetic','Kinetic to Potential','Potential to Kinetic']
plt.plot(time,Ex["KineticToInternal"].data)
plt.plot(time,Ex["InternalToKinetic"].data)
plt.plot(time,Ex["KineticToPotential"].data)
plt.plot(time,Ex["PotentialToKinetic"].data)
plt.legend(ex_names)
plt.xlabel('time (days)')
plt.show()

Now lets compute a 1D global vertical profile of the energy components at each time level by preserving the vertical "level" dimension:

In [ ]:
sub_domain_longitude = np.array([None,None])
sub_domain_latitude = np.array([None,None])
E = energy_components(ds, fieldnames, sub_domain_longitude, sub_domain_latitude, preserve_dims="level")

In [ ]:
plt.semilogx(E["Internal"].sel(time=E.time[-1]),E.level)
plt.semilogx(E["Latent"].sel(time=E.time[-1]),E.level)
plt.semilogx(E["HorizontalKinetic"].sel(time=E.time[-1]),E.level)
plt.semilogx(E["VerticalKinetic"].sel(time=E.time[-1]),E.level)
plt.gca().invert_yaxis()
plt.legend(['Internal','Latent','Kinetic (horiz.)','Kinetic (vert.)'])
plt.ylabel('pressure, hPa')
plt.show()

Conversely, we can also intergrate in the vertical only, in order to genreate a 2D horizontal plot of the different energy components at each time level:

In [ ]:
E = energy_components(ds, fieldnames, sub_domain_longitude, sub_domain_latitude, preserve_dims=["longitude","latitude"])

In [ ]:
plt.rcParams['image.cmap'] = 'jet'
lat2d,lon2d=np.meshgrid(E.latitude.data,E.longitude.data)
plt.contourf(lon2d,lat2d,E["Latent"].sel(time=E.time[-1]).data,100)
plt.show()

In [ ]:
plt.contourf(lon2d,lat2d,E["HorizontalKinetic"].sel(time=E.time[-1]).data,100)
plt.show()

We can also generate vertical profiles of the energy exchanges, integrated over the horizontal and over time:

In [ ]:
Ex = energy_exchanges(ds, fieldnames, sub_domain_longitude, sub_domain_latitude,preserve_dims="level",reduce_dims="time")

In [ ]:
plt.plot(Ex["KineticToInternal"],Ex.level)
plt.plot(Ex["InternalToKinetic"],Ex.level)
plt.plot(Ex["KineticToPotential"],Ex.level)
plt.plot(Ex["PotentialToKinetic"],Ex.level)
plt.gca().invert_yaxis()
plt.legend(['Kinetic to Internal','Internal to Kinetic','Kinetic to Potential','Potential to Kinetic'])
plt.ylabel('pressure, hPa')
plt.show()

In [ ]:
Ex = energy_exchanges(ds, fieldnames, sub_domain_longitude, sub_domain_latitude, preserve_dims=["latitude","longitude"], reduce_dims="time")

In [ ]:
plt.rcParams['image.cmap'] = 'jet'
lat2d,lon2d=np.meshgrid(Ex.latitude.data,Ex.longitude.data)
plt.contourf(lon2d,lat2d,Ex["KineticToInternal"].data,100)
plt.show()